In [1]:
# ---
# jupyter:
#   jupytext:
#     text_representation:
#       extension: .py
#       format_name: percent
#       format_version: '1.3'
#       jupytext_version: 1.17.1
#   kernelspec:
#     display_name: Python 3 (ipykernel)
#     language: python
#     name: python3
# ---


In [2]:
%load_ext autoreload
%autoreload 2


Failed to read module file 'C:\Users\LENOVO\miniconda3\envs\dl-pi1\Lib\functools.py' for module 'functools': UnicodeDecodeError
Traceback (most recent call last):
  File "C:\Users\LENOVO\miniconda3\envs\dl-pi1\Lib\site-packages\IPython\core\extensions.py", line 62, in load_extension
    return self._load_extension(module_str)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\LENOVO\miniconda3\envs\dl-pi1\Lib\site-packages\IPython\core\extensions.py", line 77, in _load_extension
    mod = import_module(module_str)
          ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\LENOVO\miniconda3\envs\dl-pi1\Lib\importlib\__init__.py", line 126, in import_module
    return _bootstrap._gcd_import(name[level:], package, level)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "<frozen importlib._bootstrap>", line 1204, in _gcd_import
  File "<frozen importlib._bootstrap>", line 1176, in _find_and_load
  File "<frozen importlib._bootstrap>", line 1140, in _find_and_load_

In [3]:
import torch
from lightning import Trainer
from lightning.pytorch.loggers import TensorBoardLogger
from lightning.pytorch.callbacks.early_stopping import EarlyStopping

from a03_functions import (
    ReviewsDataset,
    ReviewsDataModule,
    LitSimpleLSTM,
)

from a03_helper import DEVICE



C:\Users\LENOVO\miniconda3\envs\dl-pi1\Lib\site-packages\torchtext\data\__init__.py:4: UserWarning: 
/!\ IMPORTANT WARNING ABOUT TORCHTEXT STATUS /!\ 
Torchtext is deprecated and the last released version will be 0.18 (this one). You can silence this warning by calling the following at the beginnign of your scripts: `import torchtext; torchtext.disable_torchtext_deprecation_warning()`
  warnings.warn(torchtext._TORCHTEXT_DEPRECATION_MSG)
Seed set to 0


In [4]:
# Initialize the data module.
dataset = ReviewsDataset(use_vocab=True)
dm = ReviewsDataModule(dataset)

# Initialize the model.
vocab_size = len(dm.vocab)
embedding_dim = 128
hidden_dim = 128
num_layers = 2
n_epochs = 10
cell_dropout = 0.5
model = LitSimpleLSTM(vocab_size, embedding_dim, hidden_dim, num_layers, cell_dropout)


In [5]:
# Test the model's forward method.
dm.setup("fit")
train_loader = dm.train_dataloader()
reviews, labels = next(iter(train_loader))
with torch.no_grad():
    out = model(reviews)

print(out)
print(tuple(t.shape for t in out))

# Should give you something such as (conrete numbers may differ):
# (
# tensor([[0.4875], [0.4942], [0.4918], ..., device='cuda:0'),
# tensor([[-0.0833, -0.0235, 0.0280, ..., device='cuda:0')
# )
# Shape:
#
# tuple[torch.Size([32, 1]), torch.Size([32, 100])]


(tensor([[0.5214],
        [0.4983],
        [0.5123],
        [0.5241],
        [0.5150],
        [0.5185],
        [0.5139],
        [0.5231],
        [0.5204],
        [0.5003],
        [0.4959],
        [0.4954],
        [0.5184],
        [0.5020],
        [0.5116],
        [0.5120],
        [0.5045],
        [0.5179],
        [0.4956],
        [0.5240],
        [0.5042],
        [0.5142],
        [0.5223],
        [0.5030],
        [0.5223],
        [0.5194],
        [0.5137],
        [0.5013],
        [0.4955],
        [0.5137],
        [0.5128],
        [0.5188]]), tensor([[-0.0221,  0.0725, -0.1440,  ...,  0.0095,  0.0699, -0.0656],
        [-0.1187,  0.0970, -0.0755,  ..., -0.0075, -0.0035,  0.0591],
        [-0.0774,  0.0858, -0.1907,  ...,  0.0112,  0.1270, -0.0556],
        ...,
        [-0.0183,  0.0518, -0.2269,  ...,  0.0288,  0.1888,  0.0385],
        [-0.0087,  0.1266, -0.1421,  ..., -0.0459,  0.0735,  0.0599],
        [-0.0865,  0.1167, -0.1583,  ..., -0.0404,  0.0597

In [15]:
# Initialize the trainer.
# TODO: YOUR CODE HERE
logger = TensorBoardLogger('logs', name = 'simple_lstm')
trainer = Trainer(
    logger=logger,
    max_epochs=n_epochs,
    check_val_every_n_epoch=10,
    gradient_clip_val=3,
)

GPU available: False, used: False
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


In [16]:
# Train the model.
trainer.fit(model, datamodule=dm)

# Result after 10 epochs (roughly):
# val_loss=0.793, val_acc=0.726, train_acc=0.956


💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.

  | Name  | Type       | Params | Mode  | FLOPs
-----------------------------------------------------
0 | model | SimpleLSTM | 4.4 M  | train | 0    
-----------------------------------------------------
4.4 M     Trainable params
0         Non-trainable params
4.4 M     Total params
17.627    Total estimated model params size (MB)
5         Modules in train mode
0         Modules in eval mode
0         Total Flops


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

`Trainer.fit` stopped: `max_epochs=10` reached.


In [17]:
trainer.callback_metrics

{'train_loss': tensor(0.0202),
 'train_loss_step': tensor(0.0026),
 'train_loss_epoch': tensor(0.0202),
 'train_acc': tensor(0.9941),
 'val_loss': tensor(1.0556),
 'val_acc': tensor(0.7750)}

In [18]:
# Evaluate.
trainer.test(model, datamodule=dm)

# Should give you something similar to:
# [{'test_loss': 0.695..., 'test_acc': 0.737...}]


Testing: |          | 0/? [00:00<?, ?it/s]

────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
       Test metric             DataLoader 0
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
        test_acc             0.800000011920929
        test_loss           0.8233492374420166
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────


[{'test_loss': 0.8233492374420166, 'test_acc': 0.800000011920929}]